In [5]:
import torch
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from torch import Tensor
from tqdm import trange

from datasets import CubeObstacle, CylinderObstacle
from utils.tools import calc_sig_strength_gpu
from utils.config import Hyperparameters as hp

random_seed = 42

In [6]:
# define the obstacles

# Create obstacles and convert to torch tensors

torch.manual_seed(random_seed)
np.random.seed(random_seed)
if hp.device == "cuda":
    torch.cuda.manual_seed_all(random_seed)

obstacle_ls = [
    CubeObstacle(-30, 25, 35, 60, 20, 0.1),
    CubeObstacle(-30, -25, 45, 10, 35, 0.1),
    CubeObstacle(-30, -60, 35, 60, 20, 0.1),
    CubeObstacle(50, -20, 35, 25, 25, 0.1),
    CylinderObstacle(10, -5,  70, 15, 0.1),
]

obst_points = []
for obstacle in obstacle_ls:
    obst_points.append(torch.tensor(obstacle.points, dtype=torch.float32))

obst_points = torch.cat([op for op in obst_points], dim=1).mT.to(hp.device)

In [7]:
def brute_force_blockage(gn_tensor: Tensor, height: float = 70, chunk_size: int = 1000, device: str = 'cpu'):
    x = torch.arange(-100, 100.01, 0.01, device=device)
    y = torch.arange(-100, 100.01, 0.01, device=device)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    Z = torch.full_like(X, height, device=device)
    grid = torch.stack([X, Y, Z], dim=-1)
    n, m, _ = grid.shape

    results = []

    for i in trange(0, n, chunk_size):
        grid_chunk = grid[i: i + chunk_size].reshape(-1, 3)
        result_se = calc_sig_strength_gpu(grid_chunk, gn_tensor, obst_points)
        results.append(result_se.cpu())

    result_se = torch.cat(results, dim=0)
    return result_se

In [8]:
# Performance evaluation base line (gn_num test)

gn_test = torch.tensor([[60, 50, 0], [35, 10, 0], [-14, 60, 0], [-50, 0, 0]], dtype=torch.float32, device=hp.device)
result_se = brute_force_blockage(gn_test, device=hp.device, chunk_size=2)
result_se

RuntimeError: MPS backend out of memory (MPS allocated: 17.32 GB, other allocations: 2.84 MB, max allowed: 20.40 GB). Tried to allocate 4.47 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).